# 07 · Complementos para el reporte final

**Qué responde:** lo que decidimos sumar al reporte el 20 oct, todo recalculado con **un solo marco** (`src/framework.py`)
y la **ventana oficial** (Apertura 2021 – Clausura 2025):
- **Recuperación tras pérdida medida por evento** (≤ 5 s después de perder el balón): la medición por posesión la subestimaba.
- **Tipos de gol**: cada gol sin penal, desde que el América tuvo el balón hasta el remate. El balón parado (definición del
  marco) va aparte; el resto se resume con un PCA (análisis de componentes principales) y se agrupa con k-means.
- **Recetas de gol**: origen de la jugada (marco) × jugada previa al remate.
- **Defensa bajo ataque**: cuando el rival llega a nuestras últimas 30 yardas, ¿cómo termina la jugada?
- **Finales y rivales**: desde el cubo por partido.

A disco solo van agregados (`story.save_story`); nunca eventos.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display

from src import cube as K, hipotesis as H, story as S
from src.pipeline.config import GOLD

pd.set_option("display.max_columns", 30, "display.width", 250, "display.float_format", lambda v: f"{v:,.2f}")
scope = S.match_scope(pd.read_parquet(GOLD / "dim_match.parquet"))
ev = S.load_story_events(scope)            # en memoria; nunca se guarda
poss = S.load_possessions(scope)
print(scope.coach.value_counts().to_dict())

{'Jardine': 91, 'Ortiz': 55, 'Solari': 27}


## Recuperación tras pérdida, por evento

In [2]:
cp = S.counterpress_events_summary(ev, scope)
display(cp); S.save_story(cp, "contrapresion_evento")

,coach,perdidas,recuperadas,perdidas_por_partido,pct_recuperadas,pct_recuperadas_Medio,pct_recuperadas_Propio,pct_recuperadas_Último
0,Solari,2488,612,92.15,24.60,24.77,21.77,25.79
1,Ortiz,5118,1256,93.05,24.54,23.14,18.60,27.72
2,Jardine,8423,2071,92.56,24.59,25.00,20.02,26.31


'C:\\Users\\pauli\\Documents\\ISAC\\ISAC-Hackathon-America\\data\\gold\\analysis\\story_contrapresion_evento.parquet'

## Tipos de gol

Rasgos de cada jugada de gol: dónde empieza (x, yardas desde nuestra portería), cuánto dura, pases, conducciones, jugadores
que la tocan, cuánto avanza, a qué velocidad, pases largos y acciones progresivas. Se estandarizan, se resumen con un PCA
(dos componentes) y se agrupan en 3 tipos con k-means; el balón parado del marco es el cuarto tipo.

In [3]:
gs = S.goal_sequences(ev, poss)
gs, model = S.goal_types(gs)
tipos, reparto = S.goal_type_summary(gs)
recetas = S.goal_recipes(gs)
cargas = model["cargas"].reset_index(names="rasgo").assign(varianza_pc1=model["varianza"][0], varianza_pc2=model["varianza"][1])
display(tipos, reparto, recetas, cargas)
S.save_story(tipos, "goles_tipos"); S.save_story(reparto, "goles_reparto"); S.save_story(recetas, "goles_recetas")
S.save_story(cargas, "goles_pca_cargas")
S.save_story(gs.loc[gs.tipo != "Balón parado", ["coach", "tipo", "PC1", "PC2", "xg"]].reset_index(drop=True), "goles_pca_puntos")
print(f"{len(gs)} goles sin penal; varianza explicada: PC1 {model['varianza'][0]:.0%}, PC2 {model['varianza'][1]:.0%}")

,tipo,goles,pct,mediana_inicio_x,mediana_duracion_s,mediana_pases,mediana_jugadores,pct_transicion_marco,xg_medio
0,Ataque elaborado,43,15.64,34.40,48.00,15.00,9.00,27.91,0.28
1,Ataque corto en campo rival,57,20.73,76.50,12.00,3.00,4.00,71.93,0.25
2,Ataque rápido tras recuperar,83,30.18,30.70,15.00,4.00,5.00,60.24,0.29
3,Balón parado,92,33.45,93.60,9.00,3.00,4.00,0.00,0.27


tipo,coach,Ataque elaborado,Ataque corto en campo rival,Ataque rápido tras recuperar,Balón parado,goles
0,Solari,7.69,19.23,46.15,26.92,26
1,Ortiz,15.69,19.61,29.41,35.29,102
2,Jardine,17.01,21.77,27.89,33.33,147


,origin,assist,goles,pct
0,Transición,Otro pase,32,11.64
1,Juego posicional,Otro pase,28,10.18
2,Juego posicional,Centro,23,8.36
3,Balón parado,Rebote o balón suelto,22,8.00
4,Transición,Rebote o balón suelto,22,8.00
5,Balón parado,Centro,17,6.18
6,Transición,Centro,15,5.45
7,Balón parado,Otro pase,14,5.09


,rasgo,PC1,PC2,varianza_pc1,varianza_pc2
0,inicio_x,-0.20,-0.58,0.54,0.24
1,duracion_s,0.43,-0.09,0.54,0.24
2,pases,0.43,-0.12,0.54,0.24
3,conducciones,0.42,-0.13,0.54,0.24
4,jugadores,0.43,-0.02,0.54,0.24
5,avance,0.21,0.58,0.54,0.24
6,velocidad,-0.22,0.52,0.54,0.24
7,pases_largos,0.29,-0.02,0.54,0.24
8,progresivos,0.23,0.14,0.54,0.24


275 goles sin penal; varianza explicada: PC1 54%, PC2 24%


## Defensa bajo ataque: las últimas 30 yardas

In [4]:
dba = S.defense_under_attack(ev, poss, scope)
display(dba); S.save_story(dba, "defensa_bajo_ataque")
del ev

,coach,llegadas,pct_sin_tiro,pct_tiro,pct_gol,xg_por_llegada,llegadas_por_partido
0,Solari,1152,79.69,20.31,2.00,0.02,42.67
1,Ortiz,2448,78.10,21.90,2.00,0.02,44.51
2,Jardine,4061,77.20,22.80,1.70,0.02,44.63


## Finales y rivales (cubo por partido, ventana)

In [5]:
C = K.load(); P = C["partidos"]; PV = P[P.muestra == "ventana"]
s = H.series_liguilla(PV, C["balon_parado"])
fin = s[s.ronda == "Final"][["torneo", "tecnico", "rival", "global", "avanzo", "posesion", "ppda", "xg_favor", "xg_contra"]]
reg = PV[(PV.coach == "Jardine") & (PV.fase == "Fase regular")]
ref = pd.DataFrame([{"torneo": "Referencia", "tecnico": "Jardine", "rival": "Fase regular (todos)", "posesion": reg.possession_time_pct.mean(),
                     "ppda": reg.ppda.median()},
                    {"torneo": "Referencia", "tecnico": "Jardine", "rival": "Fase regular contra rivales fuertes",
                     "posesion": reg[reg.rival_nivel == "Fuerte"].possession_time_pct.mean(), "ppda": reg[reg.rival_nivel == "Fuerte"].ppda.median()}])
finales = pd.concat([fin, ref], ignore_index=True)
riv = K.rivales(PV[PV.coach.isin(H.COACHES)], min_partidos=5)
display(finales, riv)
S.save_story(finales, "finales"); S.save_story(riv, "rivales")

,torneo,tecnico,rival,global,avanzo,posesion,ppda,xg_favor,xg_contra
0,Apertura 2023,Jardine,Tigres UANL,4-1,True,50.17,14.41,2.52,1.21
1,Clausura 2024,Jardine,Cruz Azul,2-1,True,43.85,30.27,2.28,1.73
2,Apertura 2024,Jardine,Monterrey,3-2,True,43.05,28.80,2.08,1.23
3,Clausura 2025,Jardine,Toluca,0-2,False,59.13,10.41,0.87,1.25
4,Referencia,Jardine,Fase regular (todos),NaN,NaN,55.06,13.12,NaN,NaN
5,Referencia,Jardine,Fase regular contra rivales fuertes,NaN,NaN,49.19,17.62,NaN,NaN


,opponent,partidos,puntos,xpts,pct_victorias,posesion,ppda,xg_favor,xg_contra,nivel,diferencia
0,Cruz Azul,14,1.79,1.57,50.00,48.17,15.20,0.97,0.74,Fuerte,0.21
1,Toluca,14,1.57,1.49,42.86,52.96,13.94,1.52,1.03,Fuerte,0.08
2,Pachuca,14,1.21,1.62,28.57,53.18,10.74,1.30,0.96,Fuerte,-0.40
3,Puebla,12,2.25,2.11,66.67,54.61,11.76,1.83,0.76,Débil,0.14
4,Atlético San Luis,12,2.00,1.62,66.67,54.73,12.59,1.40,1.06,Medio,0.38
5,Guadalajara,12,1.92,1.47,50.00,49.98,13.06,0.98,0.87,Fuerte,0.45
6,Tigres UANL,10,2.30,1.81,70.00,45.25,17.15,1.24,0.89,Fuerte,0.49
7,León,10,1.50,1.82,30.00,54.98,14.84,1.28,0.93,Medio,-0.32
8,Monterrey,10,1.50,1.62,40.00,50.26,15.01,1.05,0.75,Fuerte,-0.12
9,Pumas UNAM,10,1.50,1.74,40.00,55.38,10.49,1.02,0.80,Medio,-0.24


'C:\\Users\\pauli\\Documents\\ISAC\\ISAC-Hackathon-America\\data\\gold\\analysis\\story_rivales.parquet'